# 3 · Results, stability & learning curve

Pulls together the saved model outputs into the cross-model comparison, then
runs the two robustness experiments:

- **Stability** — 20 group-splits with a Nadeau-Bengio corrected paired t-test,
  so we know whether a model's lead is real or seed noise.
- **Learning curve** — F1 vs training-set size on a fixed test set, 5 seeds.

Run notebook **2** first for the per-model result files.

In [ ]:
# Make the credit_rating package importable whether or not it's pip-installed.
try:
    import credit_rating  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path.cwd().parent))

from credit_rating import config, data, models, evaluate, experiments, plots
print("Package loaded. Device:", config.get_device())

## 3.1 · Cross-model comparison

In [ ]:
import joblib, pandas as pd
rows = []
for fname in ["xgb_results.pkl", "tabpfn3_results.pkl", "autogluon_results.pkl"]:
    p = config.RESULTS_DIR / fname
    if p.exists():
        r = joblib.load(p)
        rows.append({"model": r["model"], "accuracy": r["accuracy"],
                     "f1_macro": r["f1_macro"], "f1_weighted": r["f1_weighted"]})
comp = pd.DataFrame(rows).set_index("model")
print(comp.round(4))

In [ ]:
import matplotlib.pyplot as plt
ax = comp[["accuracy", "f1_macro", "f1_weighted"]].plot(
    kind="bar", figsize=(9, 5), edgecolor="white",
    color=["#8da0cb", "#fc8d62", "#66c2a5"])
ax.set_title("Model comparison"); ax.set_ylabel("Score")
ax.set_ylim(0, 1); plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "model_comparison.png", dpi=150); plt.show()

## 3.2 · AutoGluon per-class report & confusion matrix

Loads the saved best-config predictions — no retraining.

In [ ]:
import joblib
X_train, X_test, y_train, y_test = data.load_splits()
ag = joblib.load(config.RESULTS_DIR / "autogluon_results.pkl")
evaluate.print_classification_report(y_test, ag["y_pred"], "AutoGluon - per-class report")
evaluate.plot_confusion_matrix(y_test, ag["y_pred"], "AutoGluon", cmap="Greens",
                               save_path=config.FIGURES_DIR / "ag_confusion_matrix.png")

## 3.3 · Stability analysis

**Slow** (~15-20 min without AutoGluon, much longer with). Set `run_ag=False` for
a quick pass. Results are cached to `results/stability_runs.csv`.

In [ ]:
df = data.preprocess(data.load_raw())
X, y, groups = data.get_xy_groups(df)

cache = config.RESULTS_DIR / "stability_runs.csv"
if cache.exists():
    import pandas as pd
    res = pd.read_csv(cache)
    print("Loaded cached stability runs.")
else:
    res = experiments.run_stability_analysis(X, y, groups, n_splits=20, run_ag=True)

experiments.summarize_stability(res)
plots.plot_stability_box(res, save_path=config.FIGURES_DIR / "stability_boxplot.png")

## 3.4 · Learning curve

**Slow.** Set `run_ag=False` to skip AutoGluon. Cached to
`results/learning_curve_results.csv`.

In [ ]:
cache = config.RESULTS_DIR / "learning_curve_results.csv"
if cache.exists():
    import pandas as pd
    lc_df = pd.read_csv(cache)
    print("Loaded cached learning-curve runs.")
else:
    lc_df = experiments.run_learning_curve(X, y, groups, run_ag=True)

plots.plot_learning_curve(lc_df, save_path=config.FIGURES_DIR / "learning_curve.png")